# Setup

In [1]:
# ! pip install -r requirements.txt
# ! playwright install chromium

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field
from typing import List, Optional   
import agentops
import os, time, json, webbrowser
from dotenv import load_dotenv
from scrapegraph_py import ScrapeGraphAI, FetchConfig

load_dotenv()

In [ ]:
os.environ["GROQ_API_KEY"]     = os.getenv("GROQ_API_KEY")
os.environ["AGENTOPS_API_KEY"] = os.getenv("AGENTOPS_API_KEY")
os.environ["COHERE_API_KEY"]   = os.getenv("COHERE_API_KEY")

sgai = ScrapeGraphAI(api_key=os.getenv("SGAI_API_KEY"))

agentops.init(
    api_key=os.getenv("AGENTOPS_API_KEY"),
    skip_auto_end_session=True
)

In [ ]:
output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

queries_llm = LLM(
    model="cohere/command-a-plus-05-2026",
    api_key=os.getenv("COHERE_API_KEY"),
    temperature=0
)

report_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# Variables

In [ ]:
product_name       = "iPhone 17 Pro"
website_list       = ["amazon.com", "ebay.com", "noon.com"]
country_name       = "USA"
num_keywords       = 3
language           = "English"
max_urls_to_scrape = 6

# Agent A: Search Queries

In [ ]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title="Search Queries", min_length=3)

search_queries_recommendation_agent = Agent(
    role="Search Queries Recommendation Agent",
    goal="Generate {num_keywords} effective search queries for {product_name}.",
    backstory="Expert SEO strategist focused on e-commerce keyword research.",
    llm=queries_llm,
    verbose=True
)

search_queries_recommendation_task = Task(
    description=(
        "Generate {num_keywords} search queries for {product_name} "
        "targeting {website_list} in {country_name}. Language: {language}."
    ),
    expected_output="A JSON object containing a list of search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "Step_1_suggested_search_queries.json"),
    agent=search_queries_recommendation_agent
)

# Search (ScrapeGraphAI)

In [ ]:
PRODUCT_URL_MARKERS = ["/itm/", "/dp/", "/p/", "/gp/product/"]

def is_product_url(url: str) -> bool:
    return any(m in url for m in PRODUCT_URL_MARKERS)


def run_search(queries, max_total=3):
    all_results, seen = [], set()
    for q in queries:
        try:
            resp = sgai.search(
                query=q,
                num_results=5,
                prompt="Find e-commerce product pages for the searched item.",
            )
            items = getattr(resp.data, "results", None) or []
            for it in items:
                d = it.model_dump() if hasattr(it, "model_dump") else dict(it)
                url = d.get("url", "")
                if not url or url in seen:
                    continue
                seen.add(url)
                all_results.append({
                    "title": (d.get("title") or "")[:100],
                    "url": url,
                    "search_query": q,
                    "is_product": is_product_url(url),
                })
        except Exception as e:
            print(f"search error for {q}: {e}")

    products = [r for r in all_results if r["is_product"]]
    chosen = (products or all_results)[:max_total]
    with open(os.path.join(output_dir, "Step_2_search_results.json"), "w") as f:
        json.dump({"results": chosen}, f, indent=2)
    return chosen

# Extract Prices (ScrapeGraphAI)

In [ ]:
def fetch_product_data(page_url: str) -> dict:
    schema = {
        "title": "string",
        "price": "number",
        "currency": "string",
        "availability": "string",
    }
    try:
        resp = sgai.extract(
            prompt=(
                "Extract details for the MAIN product on this page ONLY - "
                "the single primary item this page is selling. "
                "IGNORE related products, similar items, accessories, cases, chargers. "
                "Return exactly ONE product: title, price (number only), currency, availability. "
                "If a field is missing, set it to null."
            ),
            url=page_url,
            schema=schema,
            fetch_config=FetchConfig(
                mode="js",
                stealth=True,
                wait=3000,
                country="us",
                scrolls=0,
            ),
        )
        return resp.data.json_data or {}
    except Exception as e:
        return {"error": str(e)}


def prescrape(max_urls=3):
    with open(os.path.join(output_dir, "Step_2_search_results.json")) as f:
        results = json.load(f).get("results", [])[:max_urls]

    scraped = []
    for r in results:
        url = r.get("url", "")
        if not url:
            continue
        product = fetch_product_data(url)
        website = url.split("/")[2] if "//" in url else url
        if "error" not in product and product.get("price"):
            scraped.append({"page_url": url, "website": website, **product})
        time.sleep(1)

    with open(os.path.join(output_dir, "Step_3_scraped_products.json"), "w") as f:
        json.dump({"products": scraped}, f, indent=2)
    return scraped

# Agent B: HTML Report

In [ ]:
report_agent = Agent(
    role="Price Comparison Report Agent",
    goal="Build a clean HTML price-comparison report for {product_name}.",
    backstory=(
        "Front-end specialist who turns structured product data into a polished, "
        "minimal, responsive HTML comparison page."
    ),
    llm=report_llm,
    verbose=True
)

report_task = Task(
    description=(
        "You are given JSON product data for {product_name}.\n"
        "Fill in the EXACT HTML template below with real data from the JSON.\n"
        "Rules:\n"
        "- Replace ALL placeholders like PRODUCT_NAME, TOTAL, LOWEST, HIGHEST, AVERAGE, TABLE_ROWS with real values.\n"
        "- TABLE_ROWS: generate one <tr> per product, sorted by price ascending.\n"
        "- The cheapest row gets class='best-row' and a <span class='badge'>Best Price</span>.\n"
        "- Prices formatted as $X,XXX.XX\n"
        "- Link column: <a class='btn' href='PAGE_URL' target='_blank'>View</a>\n"
        "- Output ONLY the filled HTML. No markdown. No code fences. No explanation.\n\n"
        "Product data JSON:\n{products_json}\n\n"
        "HTML TEMPLATE TO FILL:\n"
        "<!DOCTYPE html>\n"
        "<html lang='en'>\n"
        "<head>\n"
        "<meta charset='UTF-8'>\n"
        "<meta name='viewport' content='width=device-width, initial-scale=1.0'>\n"
        "<title>Price Comparison - PRODUCT_NAME</title>\n"
        "<style>\n"
        "  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}\n"
        "  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #f4f6fb; color: #1a1a2e; min-height: 100vh; padding: 40px 20px; }}\n"
        "  .container {{ max-width: 1100px; margin: 0 auto; }}\n"
        "  header {{ margin-bottom: 32px; }}\n"
        "  header h1 {{ font-size: 1.8rem; font-weight: 700; color: #0d1b4b; }}\n"
        "  header p {{ color: #666; margin-top: 6px; font-size: 0.95rem; }}\n"
        "  .cards {{ display: flex; gap: 16px; margin-bottom: 32px; flex-wrap: wrap; }}\n"
        "  .card {{ background: #fff; border-radius: 12px; padding: 20px 28px; box-shadow: 0 2px 12px rgba(0,0,0,.07); flex: 1; min-width: 160px; }}\n"
        "  .card .label {{ font-size: 0.75rem; text-transform: uppercase; letter-spacing: .08em; color: #888; margin-bottom: 6px; }}\n"
        "  .card .value {{ font-size: 1.6rem; font-weight: 700; color: #0d1b4b; }}\n"
        "  .card.best .value {{ color: #1a7f4b; }}\n"
        "  .table-wrap {{ background: #fff; border-radius: 14px; box-shadow: 0 2px 12px rgba(0,0,0,.07); overflow: hidden; }}\n"
        "  table {{ width: 100%; border-collapse: collapse; }}\n"
        "  thead tr {{ background: #0d1b4b; color: #fff; }}\n"
        "  th {{ padding: 14px 16px; text-align: left; font-size: 0.8rem; text-transform: uppercase; letter-spacing: .06em; font-weight: 600; }}\n"
        "  td {{ padding: 14px 16px; font-size: 0.9rem; border-bottom: 1px solid #f0f0f0; vertical-align: middle; }}\n"
        "  tr:last-child td {{ border-bottom: none; }}\n"
        "  tr.best-row {{ background: #f0faf5; }}\n"
        "  tr:not(.best-row):hover {{ background: #fafafa; }}\n"
        "  .badge {{ display: inline-block; background: #1a7f4b; color: #fff; font-size: 0.68rem; font-weight: 700; padding: 2px 8px; border-radius: 20px; margin-left: 8px; vertical-align: middle; text-transform: uppercase; }}\n"
        "  .btn {{ display: inline-block; background: #0d1b4b; color: #fff; padding: 6px 16px; border-radius: 6px; text-decoration: none; font-size: 0.82rem; font-weight: 600; }}\n"
        "  .btn:hover {{ background: #1a2f6b; }}\n"
        "</style>\n"
        "</head>\n"
        "<body>\n"
        "<div class='container'>\n"
        "  <header>\n"
        "    <h1>PRODUCT_NAME</h1>\n"
        "    <p>Price comparison across TOTAL listings — sorted by price (lowest first)</p>\n"
        "  </header>\n"
        "  <div class='cards'>\n"
        "    <div class='card best'><div class='label'>Lowest Price</div><div class='value'>LOWEST</div></div>\n"
        "    <div class='card'><div class='label'>Average Price</div><div class='value'>AVERAGE</div></div>\n"
        "    <div class='card'><div class='label'>Highest Price</div><div class='value'>HIGHEST</div></div>\n"
        "    <div class='card'><div class='label'>Total Listings</div><div class='value'>TOTAL</div></div>\n"
        "  </div>\n"
        "  <div class='table-wrap'>\n"
        "    <table>\n"
        "      <thead><tr><th>#</th><th>Title</th><th>Website</th><th>Price</th><th>Availability</th><th>Link</th></tr></thead>\n"
        "      <tbody>TABLE_ROWS</tbody>\n"
        "    </table>\n"
        "  </div>\n"
        "</div>\n"
        "</body>\n"
        "</html>"
    ),
    expected_output="A complete filled HTML document starting with <!DOCTYPE html>.",
    output_file=os.path.join(output_dir, "price_comparison.html"),
    agent=report_agent
)

# Run

In [ ]:
# Step 1: Generate search queries
queries_crew = Crew(
    agents=[search_queries_recommendation_agent],
    tasks=[search_queries_recommendation_task],
    process=Process.sequential,
    verbose=True
)

q_result = queries_crew.kickoff(inputs={
    "product_name": product_name,
    "website_list": website_list,
    "country_name": country_name,
    "num_keywords": num_keywords,
    "language": language,
})

queries = q_result.json_dict["queries"]
print("Queries:", queries)

In [ ]:
# Step 2: Search + extract prices
search_results = run_search(queries, max_total=max_urls_to_scrape)
print(f"Found {len(search_results)} URLs")

products = prescrape(max_urls=max_urls_to_scrape)
print(json.dumps(products, indent=2, default=str))

In [ ]:
# Step 3: Generate HTML report
report_crew = Crew(
    agents=[report_agent],
    tasks=[report_task],
    process=Process.sequential,
    verbose=True
)

report_crew.kickoff(inputs={
    "product_name": product_name,
    "products_json": json.dumps(products, ensure_ascii=False)
})

In [ ]:
# Step 4: Clean HTML + open in browser
html_path = os.path.join(output_dir, "price_comparison.html")

with open(html_path) as f:
    html = f.read()

html = html.strip()
if html.startswith("```"):
    html = html.split("\n", 1)[1] if "\n" in html else html
    html = html.replace("```html", "").replace("```", "").strip()
for marker in ["<!DOCTYPE", "<!doctype", "<html"]:
    idx = html.find(marker)
    if idx != -1:
        html = html[idx:]
        break

with open(html_path, "w") as f:
    f.write(html)

print(f"Report saved to: {os.path.abspath(html_path)}")
webbrowser.open(f"file://{os.path.abspath(html_path)}")